In [1]:
# ============================================
# NOTEBOOK 1: DATA COLLECTION
# Building a database of African footballers
# currently playing in European football
#
# Data source: Transfermarkt (web scraping)
# Target: Player name, nationality, club,
# league, market value, age, position
# ============================================

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# Test that libraries loaded correctly
print("All libraries loaded")
print(f"Pandas version: {pd.__version__}")

All libraries loaded
Pandas version: 3.0.3


In [2]:
# ============================================
# THE 15 AFRICAN NATIONS WE WILL ANALYSE
# Selected based on historical European
# football representation and data availability
# ============================================

AFRICAN_NATIONS = {
    'Nigeria': 'nigeria',
    'Senegal': 'senegal',
    'Morocco': 'marokko',
    'Ghana': 'ghana',
    'Ivory Coast': 'elfenbeinkueste',
    'Cameroon': 'kamerun',
    'Algeria': 'algerien',
    'Egypt': 'aegypten',
    'Mali': 'mali',
    'Guinea': 'guinea',
    'DR Congo': 'dr-kongo',
    'South Africa': 'suedafrika',
    'Tunisia': 'tunesien',
    'Gabon': 'gabun',
    'Burkina Faso': 'burkina-faso'
}

print(f"Nations to analyse: {len(AFRICAN_NATIONS)}")
for nation in AFRICAN_NATIONS.keys():
    print(f" {nation}")

Nations to analyse: 15
 Nigeria
 Senegal
 Morocco
 Ghana
 Ivory Coast
 Cameroon
 Algeria
 Egypt
 Mali
 Guinea
 DR Congo
 South Africa
 Tunisia
 Gabon
 Burkina Faso


In [3]:
# ============================================
# TRANSFERMARKT SCRAPER
# Fetches African players from each nation's
# page on Transfermarkt
# ============================================

def scrape_nation_players(nation_name, tm_code):
    """
    Scrapes player data for one African nation
    from Transfermarkt's national team page
    """
    
    url = f"https://www.transfermarkt.com/{tm_code}/kader/verein/{tm_code}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        print(f"  {nation_name}: Status {response.status_code}")
        return response.status_code
    except Exception as e:
        print(f"  {nation_name}: Error — {e}")
        return None

# Test with one nation first
print("Testing connection to Transfermarkt...")
status = scrape_nation_players('Nigeria', 'nigeria')
print(f"\nStatus code 200 = success, 403 = blocked")

Testing connection to Transfermarkt...
  Nigeria: Status 200

Status code 200 = success, 403 = blocked


In [4]:
# ============================================
# FULL SCRAPER — Extract player data
# for each African nation
# ============================================

def get_nation_players(nation_name, tm_code):
    
    url = f"https://www.transfermarkt.com/{tm_code}/kader/verein/{tm_code}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        players = []
        
        # Find all player name links — these have a specific pattern
        # on Transfermarkt player profile links contain /profil/spieler/
        player_links = soup.find_all('a', href=lambda x: x and '/profil/spieler/' in x)
        
        # Also get market values
        value_cells = soup.find_all('td', {'class': 'rechts hauptlink'})
        values = [v.get_text(strip=True) for v in value_cells]
        
        seen_names = set()
        player_idx = 0
        
        for link in player_links:
            name = link.get_text(strip=True)
            
            # Skip empty names and duplicates
            if not name or name in seen_names or len(name) < 3:
                continue
                
            seen_names.add(name)
            
            # Get market value if available
            mv = values[player_idx] if player_idx < len(values) else 'Unknown'
            player_idx += 1
            
            players.append({
                'name': name,
                'nationality': nation_name,
                'market_value_raw': mv,
                'profile_url': 'https://www.transfermarkt.com' + link['href']
            })
        
        return players
        
    except Exception as e:
        print(f"  Error scraping {nation_name}: {e}")
        return []

# Test with Nigeria
print("Testing fixed scraper with Nigeria...")
nigeria_players = get_nation_players('Nigeria', 'nigeria')
print(f"Players found: {len(nigeria_players)}")
if nigeria_players:
    print("\nFirst 5 players:")
    for p in nigeria_players[:5]:
        print(f"  {p['name']} | {p['market_value_raw']}")

Testing fixed scraper with Nigeria...
Players found: 5

First 5 players:
  P. Schulze | Unknown
  A. Moreira | Unknown
  D. Pejcinovic | Unknown
  M. Cvetkovic | Unknown
  N. Brown | Unknown


In [9]:
# ============================================
# SWITCHING TO FBREF VIA SOCCERDATA
# More reliable than scraping Transfermarkt
# directly. FBref has clean player data
# including nationality, club, league, age
# ============================================

import soccerdata as sd

print("Testing soccerdata FBref connection...")

# Load FBref data for Big 5 European leagues
fbref = sd.FBref(leagues="Big 5 European Leagues", seasons="2024-2025")

# Get player stats
print("Loading player stats from FBref...")
player_stats = fbref.read_player_season_stats(stat_type="standard")

print(f"Total players loaded: {len(player_stats)}")
print(f"\nColumns available:")
print(player_stats.columns.tolist())
print(f"\nSample data:")
print(player_stats.head(3))

Testing soccerdata FBref connection...


ValueError: 
                        Invalid league 'Big 5 European Leagues'. Valid leagues are:
                        ['Big 5 European Leagues Combined',
 'ENG-Premier League',
 'ESP-La Liga',
 'FRA-Ligue 1',
 'GER-Bundesliga',
 'INT-European Championship',
 "INT-Women's World Cup",
 'INT-World Cup',
 'ITA-Serie A']
                        

In [6]:
import sys
print(sys.executable)

C:\Users\Dell\african-football-migration\venv\Scripts\python.exe


In [7]:
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'soccerdata'], 
               capture_output=True)
print("Done")

Done


In [8]:
import soccerdata as sd
print("soccerdata loaded")
print(f"Version: {sd.__version__}")

soccerdata loaded
Version: 1.9.0


In [10]:
# ============================================
# LOAD PLAYER DATA FROM FBREF
# Big 5 European Leagues Combined 2024-25
# ============================================

import soccerdata as sd
import warnings
warnings.filterwarnings('ignore')

print("Loading FBref player data...")
print("This may take 1-2 minutes...")

# Correct league name
fbref = sd.FBref(
    leagues="Big 5 European Leagues Combined", 
    seasons="2024-2025"
)

# Get standard player stats
player_stats = fbref.read_player_season_stats(stat_type="standard")

# Reset index
player_stats = player_stats.reset_index()

print(f" Total players loaded: {len(player_stats)}")
print(f"\nColumns available:")
print(player_stats.columns.tolist()[:20])

Loading FBref player data...
This may take 1-2 minutes...


[06/11/26 10:31:03] INFO     Saving cached data to C:\Users\Dell\soccerdata\data\FBref               ]8;id=1651201;file://C:\Users\Dell\african-football-migration\venv\Lib\site-packages\soccerdata\_common.py\_common.py]8;;\:]8;id=1651202;file://C:\Users\Dell\african-football-migration\venv\Lib\site-packages\soccerdata\_common.py#250\250]8;;\



*** chromedriver to download = 148.0.7778.178 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [uc_driver.exe] was saved to:
C:\Users\Dell\african-football-migration\venv\Lib\site-packages\seleniumbase\drivers\
uc_driver.exe

Making [uc_driver.exe 148.0.7778.178] executable ...
[uc_driver.exe 148.0.7778.178] is now ready for use!


*** chromedriver to download = 148.0.7778.178 (Previous Version)

https://storage.googleapis.com/chrome-for-testing-public/148.0.7778.178/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [chromedriver.exe] was saved to:
C:\Users\Dell\african-football-migration\venv\Lib\site-packages\seleniumbase\drivers\
chromedriver.exe

Making [chromedriver.exe 148.0.7778.178] executable ...
[chr

In [11]:
# ============================================
# FILTER FOR AFRICAN PLAYERS
# The 'nation' column contains nationality
# codes like 'ng NGR' for Nigeria
# We filter using African nation codes
# ============================================

# African nation codes used by FBref
AFRICAN_CODES = [
    'NGR', 'SEN', 'MAR', 'GHA', 'CIV', 'CMR',
    'ALG', 'EGY', 'MLI', 'GUI', 'COD', 'RSA',
    'TUN', 'GAB', 'BFA', 'ZIM', 'MOZ', 'ANG',
    'COG', 'BEN', 'TOG', 'GNB', 'NGA', 'ETH',
    'KEN', 'UGA', 'TAN', 'ZAM', 'MWI', 'SLE',
    'LBR', 'MDG', 'MRI', 'CPV', 'GNQ', 'COM'
]

# The nation column format is like 'ng NGR'
# so we check if any African code appears
african_players = player_stats[
    player_stats[('nation', '')].apply(
        lambda x: any(code in str(x) for code in AFRICAN_CODES)
        if pd.notna(x) else False
    )
].copy()

print(f"Total African players in Big 5 leagues: {len(african_players)}")
print(f"\nNation column sample:")
print(african_players[('nation', '')].value_counts().head(15))

Total African players in Big 5 leagues: 361

Nation column sample:
(nation, )
CIV    54
MAR    48
SEN    40
NGA    37
GHA    27
MLI    27
CMR    25
ALG    22
GUI    15
COD     9
TUN     8
EGY     7
BFA     6
ANG     6
GNB     5
Name: count, dtype: Int64


In [12]:
# ============================================
# CLEAN AND STRUCTURE AFRICAN PLAYER DATA
# ============================================

# Flatten multi-level columns
african_players.columns = [
    '_'.join(col).strip('_') if isinstance(col, tuple) 
    else col for col in african_players.columns
]

# Select and rename key columns
african_clean = african_players[[
    'player_', 'nation_', 'pos_', 'age_',
    'team_', 'league_', 'born_',
    'Performance_Gls', 'Performance_Ast',
    'Playing Time_MP', 'Playing Time_Min'
]].copy()

african_clean.columns = [
    'player', 'nation_code', 'position', 'age',
    'club', 'league', 'birth_year',
    'goals', 'assists', 'matches_played', 'minutes'
]

# Clean nation codes — extract just the 3-letter code
african_clean['nation_code'] = african_clean['nation_code'].str.extract(r'([A-Z]{3})')

# Map nation codes to full names
nation_map = {
    'CIV': 'Ivory Coast', 'MAR': 'Morocco',
    'SEN': 'Senegal', 'NGA': 'Nigeria',
    'GHA': 'Ghana', 'MLI': 'Mali',
    'CMR': 'Cameroon', 'ALG': 'Algeria',
    'GUI': 'Guinea', 'COD': 'DR Congo',
    'TUN': 'Tunisia', 'EGY': 'Egypt',
    'BFA': 'Burkina Faso', 'ANG': 'Angola',
    'GNB': 'Guinea-Bissau', 'ZIM': 'Zimbabwe',
    'MOZ': 'Mozambique', 'GAB': 'Gabon',
    'COG': 'Congo', 'BEN': 'Benin',
    'TOG': 'Togo', 'SLE': 'Sierra Leone',
    'LBR': 'Liberia', 'NGR': 'Nigeria'
}

african_clean['nationality'] = african_clean['nation_code'].map(nation_map)
african_clean['nationality'] = african_clean['nationality'].fillna(
    african_clean['nation_code']
)

# Convert age to numeric
african_clean['age'] = pd.to_numeric(
    african_clean['age'], errors='coerce'
)

# Remove rows with no player name
african_clean = african_clean.dropna(subset=['player'])
african_clean = african_clean[african_clean['player'] != '']

print(f"Clean dataset shape: {african_clean.shape}")
print(f"\nTotal African players: {len(african_clean)}")
print(f"\nTop 10 nations:")
print(african_clean['nationality'].value_counts().head(10))
print(f"\nLeague distribution:")
print(african_clean['league'].value_counts())
print(f"\nSample data:")
print(african_clean.head(5).to_string())

KeyError: "['player_', 'nation_', 'pos_', 'age_', 'team_', 'league_', 'born_'] not in index"

In [13]:
# Check actual column names after flattening
print("All columns after flattening:")
for col in african_players.columns:
    print(f"  {col}")

All columns after flattening:
  league
  season
  team
  player
  nation
  pos
  age
  born
  Playing Time_MP
  Playing Time_Starts
  Playing Time_Min
  Playing Time_90s
  Performance_Gls
  Performance_Ast
  Performance_G+A
  Performance_G-PK
  Performance_PK
  Performance_PKatt
  Performance_CrdY
  Performance_CrdR
  Per 90 Minutes_Gls
  Per 90 Minutes_Ast
  Per 90 Minutes_G+A
  Per 90 Minutes_G-PK
  Per 90 Minutes_G+A-PK


In [14]:
# ============================================
# CLEAN AND STRUCTURE AFRICAN PLAYER DATA
# ============================================

# Select and rename key columns
african_clean = african_players[[
    'player', 'nation', 'pos', 'age',
    'team', 'league', 'born',
    'Performance_Gls', 'Performance_Ast',
    'Playing Time_MP', 'Playing Time_Min'
]].copy()

african_clean.columns = [
    'player', 'nation_code', 'position', 'age',
    'club', 'league', 'birth_year',
    'goals', 'assists', 'matches_played', 'minutes'
]

# Clean nation codes — extract just the 3-letter code
african_clean['nation_code'] = african_clean['nation_code'].str.extract(r'([A-Z]{3})')

# Map nation codes to full names
nation_map = {
    'CIV': 'Ivory Coast', 'MAR': 'Morocco',
    'SEN': 'Senegal', 'NGA': 'Nigeria',
    'NGR': 'Nigeria', 'GHA': 'Ghana',
    'MLI': 'Mali', 'CMR': 'Cameroon',
    'ALG': 'Algeria', 'GUI': 'Guinea',
    'COD': 'DR Congo', 'TUN': 'Tunisia',
    'EGY': 'Egypt', 'BFA': 'Burkina Faso',
    'ANG': 'Angola', 'GNB': 'Guinea-Bissau',
    'ZIM': 'Zimbabwe', 'MOZ': 'Mozambique',
    'GAB': 'Gabon', 'COG': 'Congo',
    'BEN': 'Benin', 'TOG': 'Togo',
    'SLE': 'Sierra Leone', 'LBR': 'Liberia'
}

african_clean['nationality'] = african_clean['nation_code'].map(nation_map)
african_clean['nationality'] = african_clean['nationality'].fillna(
    african_clean['nation_code']
)

# Convert age to numeric
african_clean['age'] = pd.to_numeric(
    african_clean['age'], errors='coerce'
)

# Remove rows with no player name
african_clean = african_clean.dropna(subset=['player'])
african_clean = african_clean[african_clean['player'] != '']

print(f"Clean dataset shape: {african_clean.shape}")
print(f"\nTotal African players: {len(african_clean)}")
print(f"\nTop 10 nations:")
print(african_clean['nationality'].value_counts().head(10))
print(f"\nLeague distribution:")
print(african_clean['league'].value_counts())

Clean dataset shape: (361, 12)

Total African players: 361

Top 10 nations:
nationality
Ivory Coast    54
Morocco        48
Senegal        40
Nigeria        37
Ghana          27
Mali           27
Cameroon       25
Algeria        22
Guinea         15
DR Congo        9
Name: count, dtype: int64

League distribution:
league
FRA-Ligue 1           159
ESP-La Liga            57
ITA-Serie A            55
ENG-Premier League     54
Name: count, dtype: int64


In [15]:
# Save clean dataset
african_clean.to_csv('../data/processed/african_players_clean.csv', 
                     index=False)
print("Data saved")
print(f"\nFinal dataset: {african_clean.shape}")

# Summary observation
summary = """
DATA COLLECTION SUMMARY
=======================
Total African players in Big 5 leagues: 361
Across 12 features per player

TOP NATIONS:
1. Ivory Coast — 54 players (14.9%)
2. Morocco     — 48 players (13.3%)
3. Senegal     — 40 players (11.1%)
4. Nigeria     — 37 players (10.2%)
5. Ghana       — 27 players (7.5%)

LEAGUE DISTRIBUTION — KEY FINDING:
Ligue 1:        159 players (44.0%) ← DOMINANT
La Liga:         57 players (15.8%)
Serie A:         55 players (15.2%)
Premier League:  54 players (14.9%)

HEADLINE INSIGHT:
France (Ligue 1) has nearly 3x more African players
than England (Premier League) despite England having
a higher average wage. Historical colonial ties between
France and West/North Africa create a dominant pipeline
that no other country matches.
"""
print(summary)

Data saved

Final dataset: (361, 12)

DATA COLLECTION SUMMARY
Total African players in Big 5 leagues: 361
Across 12 features per player

TOP NATIONS:
1. Ivory Coast — 54 players (14.9%)
2. Morocco     — 48 players (13.3%)
3. Senegal     — 40 players (11.1%)
4. Nigeria     — 37 players (10.2%)
5. Ghana       — 27 players (7.5%)

LEAGUE DISTRIBUTION — KEY FINDING:
Ligue 1:        159 players (44.0%) ← DOMINANT
La Liga:         57 players (15.8%)
Serie A:         55 players (15.2%)
Premier League:  54 players (14.9%)

HEADLINE INSIGHT:
France (Ligue 1) has nearly 3x more African players
than England (Premier League) despite England having
a higher average wage. Historical colonial ties between
France and West/North Africa create a dominant pipeline
that no other country matches.

